## Config

### Install

In [0]:
%run ./00_utility

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Parameters

In [0]:
# TABLES
PREDICTIONS_TABLE = "ta_coll.whatif.out_palinsesto_predict"
HIST_TABLE = "ta_coll.whatif.storico_programmi"
DELTA_TABLE = "ta_coll.whatif.output_palinsesto_delta"


# PARAMETERS
CHANNELS = ['Rai 1', 'Rai 2', 'Rai 3']
AUDITEL_LAG_DAYS = 3
TOLERANCE_MATCH_START_TIME = 18000 # 30 minuti
START_PRIMETIME = 20 * 3600 + 30 *60 # 20.30

## Loading Palinsesto Storico e Futuro

In [0]:
# Load future programming schedule with share predictions
output = spark.table(PREDICTIONS_TABLE)
output = output.select('Data','Programma', 'Canale', 'ORA_INIZIO_TRX','orario_inizio','orario_fine','share_predetto','programma_norm','share_manuale')

output = (output.filter(F.col('Canale').isin(CHANNELS))
                            .filter(F.col('Data') >= F.current_date() - F.expr(f'INTERVAL {AUDITEL_LAG_DAYS} DAY'))
                            .select('Data','Programma', 'Canale', 'ORA_INIZIO_TRX','orario_inizio','orario_fine','share_predetto','programma_norm','share_manuale')
                            )

In [0]:
# Load historical programming 
df_programmi = spark.table(HIST_TABLE)

# Apply filters to select: - only rai TV channels - only last 2 days data
df_programmi = (df_programmi.filter(F.col('Canale').isin(CHANNELS))
                            .filter(F.col('Data') >= F.current_date() - F.expr(f'INTERVAL {AUDITEL_LAG_DAYS} DAY'))
                            .filter(F.col('ORA_INIZIO_TRX') >=  START_PRIMETIME)
                            .select('Data','programma_norm','Canale', 'ORA_INIZIO_TRX','Share')
                            )
# Convert to Pandas
df_programmi = df_programmi.toPandas()

df_programmi = spark.createDataFrame(df_programmi)

## Share Delta Calculation

In [0]:

# Aggiungiamo una colonna di supporto per tenere traccia dell'ora originale di inizio del programma prima dell'applicazione della tolleranza
df_programmi_with_hist = df_programmi.withColumn("hist_ORA_INIZIO_TRX", F.col("ORA_INIZIO_TRX"))

# Left join future programming with historical programming so that for each program we have both predicted and actual share
df_delta = output.join(df_programmi_with_hist,
                       on=
                       [
                           output["Canale"] == df_programmi_with_hist["Canale"],
                           output["Data"] == df_programmi_with_hist["Data"],
                           output["programma_norm"] == df_programmi_with_hist["programma_norm"],
                           output["ORA_INIZIO_TRX"].between(df_programmi_with_hist["ORA_INIZIO_TRX"] - TOLERANCE_MATCH_START_TIME, df_programmi_with_hist["ORA_INIZIO_TRX"] + TOLERANCE_MATCH_START_TIME)
                        ],
                       how="left").drop(*[df_programmi_with_hist[k] for k in ['Data','Canale','ORA_INIZIO_TRX','programma_norm']])

# Applicando la tolleranza potrebbero esserci dei duplicati - in questo caso teniamo il record di auditel con l'orario di inizio più vicino (anche se non esattamente uguale) a quello di tivu tivu
df_delta = df_delta.withColumn("time_diff",F.abs(F.col("ORA_INIZIO_TRX") - F.col("hist_ORA_INIZIO_TRX")))
w = Window.partitionBy("Canale", "Data", "programma_norm", "ORA_INIZIO_TRX").orderBy("time_diff")
df_delta = df_delta.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn", "time_diff", "hist_ORA_INIZIO_TRX")

# Selezionamo solo alcune colonne e calcoliamo il delta tra share previsto e manuale/reale
df_delta = (
    df_delta
    .select(
        "Canale",
        "Data",
        "Programma",
        "orario_inizio",
        "orario_fine",
        "share_predetto",
        "share_manuale",
        "Share",
    )
    .withColumnRenamed("Share", "share_reale")
    .withColumn(
        "delta_share",
        F.round(
            F.col("share_predetto") - F.coalesce(F.col("share_manuale"), F.col("share_reale")),
            4,
        ),
    )
)

## Output

In [0]:
df_delta.display()

In [0]:
df_delta = df_delta.withColumn('ID', F.concat(F.col('Canale'), F.lit('_'), F.col('Data'), F.lit('_'), F.col('Programma'), F.lit('_'), F.col('orario_inizio')))

# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(DELTA_TABLE):
    df_delta.write.format("delta").saveAsTable(DELTA_TABLE)
else:
    # otherwise do an upsert
    delta_table_comp = DeltaTable.forName(spark, DELTA_TABLE)

    (
        delta_table_comp.alias("target")
        .merge(
            df_delta.alias("source"),
            "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio AND target.Programma = source.Programma"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )